# Notebook 07 — MHJ Analysis: Human Jailbreak Detection

Applies the representation engineering pipeline to **MHJ (Multi-turn Human Jailbreaks)** — a dataset of
human-written multi-turn jailbreak conversations.

**Key difference from notebooks 04–06:** MHJ conversations already contain both user AND assistant turns.
We replay them through the model using a forward pass only (no generation), extracting hidden states
at the Zhao et al. token positions (`t_inst`, `t_post_inst`) of the final user turn.

```
... [user content] <|eot_id|> <|start_header_id|> assistant <|end_header_id|> \n\n
         ↑               ↑
      t_inst        t_post_inst
```

**Two phases:**
- **Phase 1 (Extraction):** load MHJ data → replay conversations → extract hidden states → save
- **Phase 2 (Analysis):** load saved hidden states → apply existing centroids → evaluate vs. framework results

**Centroid sources (from nb04):** crescendo, actorattack, xteaming, single_turn  
**Output:** `results/representations/mhj/`

## Phase 1: Hidden State Extraction

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Paths ─────────────────────────────────────────────────────────────────────
MHJ_DATA_PATH = repo_root / "data" / "datasets" / "MHJ - Sheet1.csv"  # ← configurable
OUT_ROOT      = repo_root / "results" / "representations" / "mhj"
CENTROID_DIR  = repo_root / "results" / "centroids"

OUT_ROOT.mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────────
N_LAYERS   = 32
HIDDEN_DIM = 4096
LAYER_INDICES = list(range(1, 33))   # hidden_states[0] = embedding, skip

SHARED_SYSTEM_PROMPT = (
    "You are a helpful, respectful and honest assistant. Always answer as helpfully "
    "as possible, while being safe."
)

print(f"repo_root:     {repo_root}")
print(f"MHJ_DATA_PATH: {MHJ_DATA_PATH}")
print(f"OUT_ROOT:      {OUT_ROOT}")
print(f"CENTROID_DIR:  {CENTROID_DIR}")

In [ ]:
# ── MHJ Data Loading ──────────────────────────────────────────────────────────
#
# FORMAT: wide CSV at data/datasets/MHJ - Sheet1.csv
#
#   Columns:
#     Source, temperature, tactic, question_id, time_spent, submission_message
#     message_0, message_1, ..., message_100
#
#   Each message_i is a JSON string: {"body": str, "role": "user"|"system"|"assistant"}
#   message_0 is always the system prompt.
#   NaN values indicate the conversation ended.
#
#   NOTE: The MHJ dataset contains only human-written USER turns — no assistant
#   turns are present. Conversations will be built with consecutive user turns
#   and no assistant responses. This means the extraction captures the model's
#   state when presented with accumulating user requests without in-context
#   assistant replies. Limitation noted in methodology.
#
#   LABEL: All 537 conversations are human-crafted jailbreaks; jailbroken=True
#   for all. The submission_message confirms what was achieved.

def load_mhj_csv(path: Path) -> list[dict]:
    """
    Load MHJ from the wide CSV format.

    Returns a list of standard-format conversation dicts:
      conv_id     : str  ("mhj_{row_index:04d}")
      goal        : str  (first user message — the opening request)
      category    : str  (tactic field)
      source      : str  (Source field)
      question_id : int
      turns       : list of {"role": str, "content": str}  — user turns only
      jailbroken  : True (all MHJ conversations achieved their goal)
      n_turns     : int  (number of user turns)
    """
    df = pd.read_csv(path)
    msg_cols = [c for c in df.columns if c.startswith("message_")]
    # Sort numerically (message_0, message_1, ..., not lexicographic)
    msg_cols.sort(key=lambda c: int(c.split("_")[1]))

    records = []
    for idx, row in df.iterrows():
        turns = []
        for c in msg_cols:
            v = row[c]
            if pd.isna(v):
                break
            try:
                m = json.loads(v)
            except (json.JSONDecodeError, TypeError):
                continue
            role = m.get("role", "")
            body = str(m.get("body", ""))
            if role == "user":
                turns.append({"role": "user", "content": body})
            # system and any assistant turns are skipped
            # (system prompt is handled separately in build_messages_mhj)

        if not turns:
            continue

        records.append({
            "conv_id":     f"mhj_{idx:04d}",
            "goal":        turns[0]["content"],    # opening user request
            "category":    str(row.get("tactic",  "")),
            "source":      str(row.get("Source",  "")),
            "question_id": row.get("question_id", None),
            "turns":       turns,
            "jailbroken":  True,                   # all MHJ are successful jailbreaks
            "n_turns":     len(turns),
        })

    return records


# ── Load ──────────────────────────────────────────────────────────────────────

if not MHJ_DATA_PATH.exists():
    print(f"MHJ data file not found: {MHJ_DATA_PATH}")
    mhj_conversations = []
else:
    mhj_conversations = load_mhj_csv(MHJ_DATA_PATH)

    n_total      = len(mhj_conversations)
    n_turns_dist = pd.Series([c["n_turns"] for c in mhj_conversations])
    tactic_dist  = pd.Series([c["category"] for c in mhj_conversations]).value_counts()
    source_dist  = pd.Series([c["source"] for c in mhj_conversations]).value_counts()

    print(f"Total conversations  : {n_total}")
    print(f"All jailbroken       : True (by construction)")
    print(f"
Turn length distribution (user turns only):")
    print(n_turns_dist.describe().to_string())
    print(f"
Tactic distribution:")
    print(tactic_dist.to_string())
    print(f"
Source distribution:")
    print(source_dist.to_string())
    print(f"
First record (standard format):")
    if mhj_conversations:
        sample = mhj_conversations[0].copy()
        sample["turns"] = sample["turns"][:2]
        print(json.dumps(sample, indent=2, default=str))


In [ ]:
# ── Model and tokenizer ───────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

# Skip if already loaded (e.g. when re-running individual cells)
if "tokenizer" not in dir() or tokenizer is None:
    print(f"Loading tokenizer: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
else:
    print("Tokenizer already loaded — skipping.")

if "model" not in dir() or model is None:
    print(f"Loading model: {MODEL_NAME}  (device_map=auto)")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    model.eval()
    print("Model loaded.")
else:
    print("Model already loaded — skipping.")

# GPU memory
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i} ({torch.cuda.get_device_name(i)}): {alloc:.2f} / {total:.2f} GB")
else:
    print("No CUDA GPUs detected — running on CPU.")

In [ ]:
# ── Extraction and replay helpers ─────────────────────────────────────────────
#
# Replay approach: for each MHJ conversation, generate assistant responses
# turn by turn so the model sees realistic context at every user turn.
#
#   replay_conversation(conv):
#     k=1 : [system, user_1]                              → extract, generate asst_1
#     k=2 : [system, user_1, asst_1, user_2]              → extract, generate asst_2
#     ...
#     k=n : [system, user_1, asst_1, ..., user_n]         → extract (no generation)
#
# This is more expensive than single forward passes but directly comparable to
# the automated framework conversations (which have real stored assistant turns).

MAX_NEW_TOKENS = 256   # tokens generated per assistant turn during replay


@torch.no_grad()
def generate_reply(model, tokenizer, messages: list[dict]) -> str:
    """
    Generate one assistant reply given messages ending with a user turn.
    Greedy decoding — fast and deterministic.
    Returns decoded text (stripped).
    """
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)

    out = model.generate(
        input_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[0, input_ids.shape[1]:]
    del out, input_ids
    torch.cuda.empty_cache()
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def get_positions(tokenizer, input_ids):
    """
    Locate t_inst and t_post_inst in tokenised input.

    Llama 3.1 Instruct template ends as:
        {user content}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n
    (add_generation_prompt=True appends the assistant header)

    t_post_inst = last <|eot_id|> position  (closes the final user turn)
    t_inst      = t_post_inst - 1           (last content token before <|eot_id|>)

    input_ids: (1, seq_len) tensor.
    """
    eot_id      = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    eot_pos     = (input_ids[0] == eot_id).nonzero(as_tuple=True)[0]
    t_post_inst = eot_pos[-1].item()
    t_inst      = t_post_inst - 1
    return t_inst, t_post_inst


@torch.no_grad()
def extract_at_positions(model, tokenizer, messages: list[dict]):
    """
    Single forward pass. Extracts hidden states at t_inst and t_post_inst.

    Returns:
      h_inst      : (32, 4096) float16 numpy
      h_post_inst : (32, 4096) float16 numpy
      t_inst      : int
      t_post_inst : int
      seq_len     : int
    """
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)

    t_inst, t_post_inst = get_positions(tokenizer, input_ids)

    outputs = model(input_ids, output_hidden_states=True)

    h_inst = np.stack([
        outputs.hidden_states[l][0, t_inst, :].cpu().to(torch.float16).numpy()
        for l in LAYER_INDICES
    ])   # (32, 4096)

    h_post = np.stack([
        outputs.hidden_states[l][0, t_post_inst, :].cpu().to(torch.float16).numpy()
        for l in LAYER_INDICES
    ])   # (32, 4096)

    del outputs, input_ids
    torch.cuda.empty_cache()

    seq_len = input_ids.shape[1]
    return h_inst, h_post, t_inst, t_post_inst, seq_len


@torch.no_grad()
def replay_conversation(model, tokenizer, conv: dict) -> list[dict] | None:
    """
    Replay a MHJ conversation with turn-by-turn assistant generation.

    At each user turn k (1-indexed):
      1. Append user_k to the growing message context.
      2. Extract hidden states (h_inst, h_post) at user_k's token positions.
      3. If k < n_turns: generate an assistant reply and append it.

    Returns a list of n_turns result dicts:
        [{
            "turn_k":      int,
            "h_inst":      (32, 4096) float16,
            "h_post":      (32, 4096) float16,
            "t_inst":      int,
            "t_post_inst": int,
            "seq_len":     int,   # input length at this turn (before generation)
        }, ...]

    Returns None if the conversation has no user turns.
    """
    user_turns = [t for t in conv["turns"] if t["role"] == "user"]
    if not user_turns:
        return None

    n = len(user_turns)
    results = []
    messages = [{"role": "system", "content": SHARED_SYSTEM_PROMPT}]

    for k, turn in enumerate(user_turns, start=1):
        messages.append({"role": "user", "content": turn["content"]})

        h_inst, h_post, t_inst, t_post_inst, seq_len = extract_at_positions(
            model, tokenizer, messages
        )
        results.append({
            "turn_k":      k,
            "h_inst":      h_inst,
            "h_post":      h_post,
            "t_inst":      t_inst,
            "t_post_inst": t_post_inst,
            "seq_len":     seq_len,
        })

        if k < n:
            reply = generate_reply(model, tokenizer, messages)
            messages.append({"role": "assistant", "content": reply})

    return results


print("Extraction and replay helpers defined.")
print(f"  MAX_NEW_TOKENS = {MAX_NEW_TOKENS}")


In [ ]:
# ── Phase 1: Replay extraction (final-turn + trajectory in one pass) ───────────
#
# For each MHJ conversation:
#   - replay_conversation() builds context turn by turn, generating assistant
#     replies between user turns.
#   - Hidden states are extracted at every user turn (trajectory).
#   - The last turn's result is also saved as the final-turn representation.
#
# Outputs:
#   OUT_ROOT / "final_turn"  / {h_inst.npy, h_post_inst.npy, metadata.parquet}
#   OUT_ROOT / "trajectory"  / {h_inst.npy, h_post_inst.npy, metadata.parquet}
#
# Resume-safe at the conversation level: completed conv_ids are skipped.

SAVE_FINAL = OUT_ROOT / "final_turn"
SAVE_TRAJ  = OUT_ROOT / "trajectory"
SAVE_FINAL.mkdir(parents=True, exist_ok=True)
SAVE_TRAJ.mkdir(parents=True, exist_ok=True)

if not mhj_conversations:
    print("No MHJ conversations loaded — skipping.")
else:
    # ── Resume: load already-done conv_ids from trajectory metadata ───────────
    traj_meta_path  = SAVE_TRAJ / "metadata.parquet"
    final_meta_path = SAVE_FINAL / "metadata.parquet"

    if traj_meta_path.exists():
        _existing_traj  = pd.read_parquet(traj_meta_path)
        done_ids = set(_existing_traj["conv_id"].unique().tolist())
        traj_h_inst_list = [np.load(str(SAVE_TRAJ / "h_inst.npy"))]
        traj_h_post_list = [np.load(str(SAVE_TRAJ / "h_post_inst.npy"))]
        traj_meta_list   = [_existing_traj]
        print(f"Resuming: {len(done_ids)} conversations already replayed.")
    else:
        done_ids         = set()
        traj_h_inst_list = []
        traj_h_post_list = []
        traj_meta_list   = []

    if final_meta_path.exists():
        final_h_inst_list = [np.load(str(SAVE_FINAL / "h_inst.npy"))]
        final_h_post_list = [np.load(str(SAVE_FINAL / "h_post_inst.npy"))]
        final_meta_list   = [pd.read_parquet(final_meta_path)]
    else:
        final_h_inst_list = []
        final_h_post_list = []
        final_meta_list   = []

    pending = [c for c in mhj_conversations if c["conv_id"] not in done_ids]
    print(f"Pending: {len(pending)} / {len(mhj_conversations)} conversations")
    total_turns = sum(c["n_turns"] for c in pending)
    print(f"Total forward passes (extraction): {total_turns}")
    print(f"Total generation calls:            {sum(max(c['n_turns']-1,0) for c in pending)}")
    print()

    n_skipped = 0
    for conv in tqdm(pending, desc="Replay"):
        try:
            results = replay_conversation(model, tokenizer, conv)
        except Exception as e:
            print(f"  ERROR {conv['conv_id']}: {e}")
            n_skipped += 1
            continue

        if results is None:
            n_skipped += 1
            continue

        conv_id = conv["conv_id"]

        # ── Trajectory: all turns ─────────────────────────────────────────────
        for r in results:
            traj_h_inst_list.append(r["h_inst"][np.newaxis])   # (1, 32, 4096)
            traj_h_post_list.append(r["h_post"][np.newaxis])
            traj_meta_list.append(pd.DataFrame([{
                "conv_id":     conv_id,
                "goal":        conv["goal"],
                "category":    conv["category"],
                "source":      conv["source"],
                "jailbroken":  conv["jailbroken"],
                "n_turns":     conv["n_turns"],
                "turn_k":      r["turn_k"],
                "t_inst":      r["t_inst"],
                "t_post_inst": r["t_post_inst"],
                "seq_len":     r["seq_len"],
            }]))

        # ── Final turn: last result only ──────────────────────────────────────
        last = results[-1]
        final_h_inst_list.append(last["h_inst"][np.newaxis])
        final_h_post_list.append(last["h_post"][np.newaxis])
        final_meta_list.append(pd.DataFrame([{
            "conv_id":     conv_id,
            "goal":        conv["goal"],
            "category":    conv["category"],
            "source":      conv["source"],
            "jailbroken":  conv["jailbroken"],
            "n_turns":     conv["n_turns"],
            "t_inst":      last["t_inst"],
            "t_post_inst": last["t_post_inst"],
            "seq_len":     last["seq_len"],
        }]))

        # ── Incremental save every 50 conversations ───────────────────────────
        if len(pending) > 50 and len(final_meta_list) % 50 == 0:
            _H = np.concatenate(traj_h_inst_list, axis=0).astype(np.float16)
            _P = np.concatenate(traj_h_post_list, axis=0).astype(np.float16)
            np.save(str(SAVE_TRAJ / "h_inst.npy"), _H)
            np.save(str(SAVE_TRAJ / "h_post_inst.npy"), _P)
            pd.concat(traj_meta_list, ignore_index=True).to_parquet(
                SAVE_TRAJ / "metadata.parquet", index=False)

            _H2 = np.concatenate(final_h_inst_list, axis=0).astype(np.float16)
            _P2 = np.concatenate(final_h_post_list, axis=0).astype(np.float16)
            np.save(str(SAVE_FINAL / "h_inst.npy"), _H2)
            np.save(str(SAVE_FINAL / "h_post_inst.npy"), _P2)
            pd.concat(final_meta_list, ignore_index=True).to_parquet(
                SAVE_FINAL / "metadata.parquet", index=False)

    # ── Final save ────────────────────────────────────────────────────────────
    if traj_h_inst_list:
        H_traj  = np.concatenate(traj_h_inst_list, axis=0).astype(np.float16)
        HP_traj = np.concatenate(traj_h_post_list, axis=0).astype(np.float16)
        traj_df = pd.concat(traj_meta_list, ignore_index=True)
        np.save(str(SAVE_TRAJ / "h_inst.npy"),      H_traj)
        np.save(str(SAVE_TRAJ / "h_post_inst.npy"), HP_traj)
        traj_df.to_parquet(SAVE_TRAJ / "metadata.parquet", index=False)
        print(f"\nTrajectory saved → {SAVE_TRAJ}")
        print(f"  h_inst:   {H_traj.shape}  ({H_traj.nbytes/1e9:.2f} GB)")
        print(f"  rows:     {len(traj_df)}")

    if final_h_inst_list:
        H_fin  = np.concatenate(final_h_inst_list, axis=0).astype(np.float16)
        HP_fin = np.concatenate(final_h_post_list, axis=0).astype(np.float16)
        fin_df = pd.concat(final_meta_list, ignore_index=True)
        np.save(str(SAVE_FINAL / "h_inst.npy"),      H_fin)
        np.save(str(SAVE_FINAL / "h_post_inst.npy"), HP_fin)
        fin_df.to_parquet(SAVE_FINAL / "metadata.parquet", index=False)
        print(f"\nFinal-turn saved → {SAVE_FINAL}")
        print(f"  h_inst:   {H_fin.shape}  ({H_fin.nbytes/1e9:.2f} GB)")
        print(f"  rows:     {len(fin_df)}")

    print(f"\nSkipped: {n_skipped}")


In [ ]:
# Trajectory extraction is now handled by the replay cell above (nb07-replay).
# Both final-turn and trajectory outputs are produced in a single pass.
# This cell is kept as a placeholder — nothing to run here.
print("Trajectory extraction is done in nb07-replay. See results/representations/mhj/trajectory/")


## Phase 2: Analysis

Loads the saved MHJ hidden states and applies the four centroid sources built in notebook 04
(crescendo, actorattack, xteaming, single_turn) to evaluate how well each transfers to human
jailbreak detection.

**Positive class:** `jailbroken=True`  
**Threshold:** Δ_harmful = 0 (sign test)  
**Scorer:** mean over 32 layers of `cos_sim(h_inst[l], mu_harmful[l]) - cos_sim(h_inst[l], mu_harmless[l])`

In [ ]:
# ── Load saved MHJ representations ───────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

final_h_inst     = None
final_h_post     = None
final_meta       = None
traj_h_inst_data = None
traj_h_post_data = None
traj_meta        = None

# Final turn
_final_inst_path = SAVE_FINAL / "h_inst.npy"
_final_post_path = SAVE_FINAL / "h_post_inst.npy"
_final_meta_path = SAVE_FINAL / "metadata.parquet"

if _final_inst_path.exists() and _final_post_path.exists() and _final_meta_path.exists():
    final_h_inst = np.load(str(_final_inst_path), mmap_mode="r")
    final_h_post = np.load(str(_final_post_path), mmap_mode="r")
    final_meta   = pd.read_parquet(_final_meta_path)
    print(f"Final-turn representations loaded:")
    print(f"  h_inst:  {final_h_inst.shape}  dtype={final_h_inst.dtype}")
    print(f"  h_post:  {final_h_post.shape}")
    print(f"  rows:    {len(final_meta)}")
    print(f"  jailbroken: {final_meta['jailbroken'].sum()} / {len(final_meta)}")
else:
    print(f"Final-turn representations not found at {SAVE_FINAL}.")
    print("Run Phase 1 extraction cells first.")

# Trajectory
_traj_inst_path = SAVE_TRAJ / "h_inst.npy"
_traj_post_path = SAVE_TRAJ / "h_post_inst.npy"
_traj_meta_path = SAVE_TRAJ / "metadata.parquet"

if _traj_inst_path.exists() and _traj_post_path.exists() and _traj_meta_path.exists():
    traj_h_inst_data = np.load(str(_traj_inst_path), mmap_mode="r")
    traj_h_post_data = np.load(str(_traj_post_path), mmap_mode="r")
    traj_meta        = pd.read_parquet(_traj_meta_path)
    print(f"\nTrajectory representations loaded:")
    print(f"  h_inst:  {traj_h_inst_data.shape}  dtype={traj_h_inst_data.dtype}")
    print(f"  rows:    {len(traj_meta)}")
    print(f"  turn_k range: {traj_meta['turn_k'].min()} – {traj_meta['turn_k'].max()}")
else:
    print(f"\nTrajectory representations not found at {SAVE_TRAJ}.")
    print("Run Phase 1 trajectory extraction cell first.")

In [ ]:
# ── Load centroids from nb04 and define scorers ───────────────────────────────

SOURCES = ["crescendo", "actorattack", "xteaming", "single_turn"]

centroids = {}
for source in SOURCES:
    path = CENTROID_DIR / f"{source}_centroids.pt"
    if path.exists():
        c = torch.load(path, map_location="cpu")
        centroids[source] = {k: v.numpy() for k, v in c.items()}  # (32, 4096) float32
        print(f"  {source:15s}  keys={list(c.keys())}")
    else:
        print(f"  MISSING: {path}")

print(f"\nLoaded {len(centroids)} centroid sources.")


# ── Scorer functions ──────────────────────────────────────────────────────────

def delta_harmful_batch(h_arr: np.ndarray, mu_harmful: np.ndarray, mu_harmless: np.ndarray) -> np.ndarray:
    """
    Δ_harmful scorer (mean over 32 layers of cosine similarity difference).

    h_arr      : (N, 32, 4096) float16 or float32
    mu_harmful : (32, 4096) float32
    mu_harmless: (32, 4096) float32

    Returns (N,) float32 — positive → model thinks request is harmful-topic.
    """
    h  = torch.tensor(h_arr.astype(np.float32))         # (N, 32, 4096)
    mh = torch.tensor(mu_harmful.astype(np.float32))    # (32, 4096)
    mb = torch.tensor(mu_harmless.astype(np.float32))   # (32, 4096)

    h_n  = F.normalize(h,  dim=2)                # (N, 32, 4096)
    mh_n = F.normalize(mh, dim=1).unsqueeze(0)  # (1, 32, 4096)
    mb_n = F.normalize(mb, dim=1).unsqueeze(0)

    sim_h = (h_n * mh_n).sum(dim=2)   # (N, 32)
    sim_b = (h_n * mb_n).sum(dim=2)
    return (sim_h - sim_b).mean(dim=1).numpy()  # (N,)


def proj_harmful_batch(h_arr: np.ndarray, v_harmful: np.ndarray) -> np.ndarray:
    """
    Projection scorer: project normalised h onto normalised v_harmful, average over layers.

    h_arr     : (N, 32, 4096)
    v_harmful : (32, 4096) float32

    Returns (N,) float32.
    """
    h = torch.tensor(h_arr.astype(np.float32))       # (N, 32, 4096)
    v = torch.tensor(v_harmful.astype(np.float32))   # (32, 4096)

    h_n = F.normalize(h, dim=2)
    v_n = F.normalize(v, dim=1).unsqueeze(0)
    proj = (h_n * v_n).sum(dim=2)   # (N, 32)
    return proj.mean(dim=1).numpy()  # (N,)


print("Centroids loaded and scorer functions defined.")

In [ ]:
# ── Phase 2a: Evaluate centroids on MHJ final-turn hidden states ──────────────
#
# Label: jailbroken=True → harmful (positive class, Δ > 0 should detect these)
# Threshold: 0 (sign test)

if final_h_inst is None or not centroids:
    print("Final-turn representations or centroids not available — skipping evaluation.")
else:
    h_arr  = final_h_inst[:].astype(np.float32)   # (N, 32, 4096) — load from mmap
    labels = final_meta["jailbroken"].values.astype(bool)   # True = positive

    N         = len(labels)
    n_pos     = labels.sum()
    n_neg     = N - n_pos
    print(f"N={N}  jailbroken (pos)={n_pos}  not-jailbroken (neg)={n_neg}\n")

    eval_records = []

    for source, c in centroids.items():
        if "mu_harmful" not in c or "mu_harmless" not in c:
            print(f"  {source}: missing mu_harmful or mu_harmless — skipping")
            continue

        delta = delta_harmful_batch(h_arr, c["mu_harmful"], c["mu_harmless"])  # (N,)

        preds     = delta > 0
        tp        = (preds & labels).sum()
        tn        = (~preds & ~labels).sum()
        fp        = (preds & ~labels).sum()
        fn        = (~preds & labels).sum()
        accuracy  = (tp + tn) / N
        tpr       = tp / n_pos if n_pos > 0 else float("nan")
        tnr       = tn / n_neg if n_neg > 0 else float("nan")

        mean_pos = delta[labels].mean()  if n_pos > 0 else float("nan")
        mean_neg = delta[~labels].mean() if n_neg > 0 else float("nan")

        eval_records.append({
            "centroid_source": source,
            "accuracy":        accuracy,
            "tpr":             tpr,
            "tnr":             tnr,
            "mean_delta_jailbroken":    mean_pos,
            "mean_delta_not_jailbroken": mean_neg,
        })

        # Store scores on metadata for downstream use
        final_meta[f"delta_{source}"] = delta

    eval_df = pd.DataFrame(eval_records)

    print("Detection accuracy on MHJ conversations (threshold = 0)\n")
    print(f"{'Centroid source':<18} {'Accuracy':>10} {'TPR (JB)':>10} {'TNR (!JB)':>10} "
          f"{'mean Δ JB':>11} {'mean Δ !JB':>11}")
    print("-" * 74)
    for _, row in eval_df.iterrows():
        print(
            f"{row['centroid_source']:<18} "
            f"{row['accuracy']:>10.1%} "
            f"{row['tpr']:>10.1%} "
            f"{row['tnr']:>10.1%} "
            f"{row['mean_delta_jailbroken']:>11.4f} "
            f"{row['mean_delta_not_jailbroken']:>11.4f}"
        )

    # Distribution plot
    fig, axes = plt.subplots(1, len(eval_records), figsize=(5 * len(eval_records), 4), sharey=False)
    if len(eval_records) == 1:
        axes = [axes]

    FW_PALETTE = {
        "crescendo":   "#4C72B0",
        "actorattack": "#DD8452",
        "xteaming":    "#55A868",
        "single_turn": "#C44E52",
    }

    for ax, row in zip(axes, eval_records):
        source = row["centroid_source"]
        delta  = final_meta[f"delta_{source}"].values
        col    = FW_PALETTE.get(source, "#888888")

        jb_vals  = delta[labels]
        njb_vals = delta[~labels]

        bins = np.linspace(delta.min() - 0.01, delta.max() + 0.01, 40)
        ax.hist(njb_vals, bins=bins, alpha=0.6, color="steelblue",  label="not jailbroken", density=True)
        ax.hist(jb_vals,  bins=bins, alpha=0.6, color="firebrick",  label="jailbroken",     density=True)
        ax.axvline(0, color="black", lw=1.2, ls="--", label="threshold=0")
        ax.set_title(f"{source}\nacc={row['accuracy']:.1%}", fontsize=10)
        ax.set_xlabel("Δ_harmful")
        ax.set_ylabel("Density")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle("MHJ final-turn Δ_harmful distributions by jailbreak label",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    fig_dir = repo_root / "notebooks" / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_dir / "mhj_final_turn_delta_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Phase 2b: Trajectory analysis ────────────────────────────────────────────
#
# Mean Δ_harmful at each turn k, split by jailbroken/not-jailbroken.
# Line plot with ±1 SE shading. One subplot per centroid source.

if traj_h_inst_data is None or not centroids:
    print("Trajectory representations or centroids not available — skipping trajectory analysis.")
else:
    traj_arr = traj_h_inst_data[:].astype(np.float32)   # (M, 32, 4096)
    traj_jb  = traj_meta["jailbroken"].values.astype(bool)
    traj_k   = traj_meta["turn_k"].values

    max_k    = int(traj_k.max())
    k_values = sorted(traj_meta["turn_k"].unique())

    print(f"Trajectory: M={len(traj_meta)} rows  turn_k 1–{max_k}")

    # Compute Δ scores for each centroid source
    traj_deltas = {}  # source -> (M,) array
    for source, c in centroids.items():
        if "mu_harmful" not in c or "mu_harmless" not in c:
            continue
        traj_deltas[source] = delta_harmful_batch(traj_arr, c["mu_harmful"], c["mu_harmless"])

    n_sources = len(traj_deltas)
    if n_sources == 0:
        print("No valid centroid sources for trajectory plot.")
    else:
        fig, axes = plt.subplots(1, n_sources, figsize=(5 * n_sources, 4), sharey=False)
        if n_sources == 1:
            axes = [axes]

        FW_PALETTE = {
            "crescendo":   "#4C72B0",
            "actorattack": "#DD8452",
            "xteaming":    "#55A868",
            "single_turn": "#C44E52",
        }

        for ax, source in zip(axes, traj_deltas):
            delta = traj_deltas[source]
            col   = FW_PALETTE.get(source, "#888888")

            mean_jb, se_jb, mean_njb, se_njb, ks_plot = [], [], [], [], []

            for k in k_values:
                mask_k   = traj_k == k
                mask_jb  = mask_k &  traj_jb
                mask_njb = mask_k & ~traj_jb

                if mask_jb.sum() < 2 and mask_njb.sum() < 2:
                    continue

                ks_plot.append(k)

                if mask_jb.sum() >= 1:
                    vals = delta[mask_jb]
                    mean_jb.append(vals.mean())
                    se_jb.append(vals.std() / np.sqrt(len(vals)) if len(vals) > 1 else 0.0)
                else:
                    mean_jb.append(float("nan"))
                    se_jb.append(0.0)

                if mask_njb.sum() >= 1:
                    vals = delta[mask_njb]
                    mean_njb.append(vals.mean())
                    se_njb.append(vals.std() / np.sqrt(len(vals)) if len(vals) > 1 else 0.0)
                else:
                    mean_njb.append(float("nan"))
                    se_njb.append(0.0)

            ks_arr     = np.array(ks_plot)
            mean_jb    = np.array(mean_jb)
            se_jb      = np.array(se_jb)
            mean_njb   = np.array(mean_njb)
            se_njb     = np.array(se_njb)

            ax.plot(ks_arr, mean_jb,  lw=2,   color="firebrick",  label="jailbroken")
            ax.fill_between(ks_arr, mean_jb - se_jb, mean_jb + se_jb,
                            alpha=0.25, color="firebrick")

            ax.plot(ks_arr, mean_njb, lw=2,   color="steelblue",  label="not jailbroken")
            ax.fill_between(ks_arr, mean_njb - se_njb, mean_njb + se_njb,
                            alpha=0.25, color="steelblue")

            ax.axhline(0, color="black", lw=0.8, ls="--")
            ax.set_title(f"Centroid: {source}", fontsize=10)
            ax.set_xlabel("Turn k")
            ax.set_ylabel("Mean Δ_harmful")
            ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)

        fig.suptitle("MHJ trajectory: mean Δ_harmful by turn (±1 SE)",
                     fontsize=12, fontweight="bold")
        plt.tight_layout()
        fig_dir = repo_root / "notebooks" / "figures"
        fig_dir.mkdir(parents=True, exist_ok=True)
        plt.savefig(fig_dir / "mhj_trajectory_delta.png", dpi=150, bbox_inches="tight")
        plt.show()

In [ ]:
# ── Summary: MHJ detection vs. within-framework accuracy (nb04 reference) ────
#
# Reference within-framework accuracy from nb04 (self-eval, test split):
#   crescendo   → 0.769
#   actorattack → 0.660
#   xteaming    → 0.594
#   single_turn → (not a multi-turn framework; used as centroid source only)

NB04_REFERENCE = {
    "crescendo":   0.769,
    "actorattack": 0.660,
    "xteaming":    0.594,
    "single_turn": None,   # no within-framework test split in nb04
}

print("=" * 72)
print("SUMMARY: Does centroid-based scoring transfer to human jailbreaks?")
print("=" * 72)
print()
print(f"{'Centroid source':<18} {'NB04 self-acc':>14} {'MHJ accuracy':>13} {'Transfer gap':>13}")
print("-" * 62)

if "eval_df" in dir() and eval_df is not None and len(eval_df) > 0:
    for _, row in eval_df.iterrows():
        source   = row["centroid_source"]
        mhj_acc  = row["accuracy"]
        nb04_acc = NB04_REFERENCE.get(source, None)

        if nb04_acc is not None:
            gap = mhj_acc - nb04_acc
            print(
                f"{source:<18} {nb04_acc:>13.1%} {mhj_acc:>13.1%} "
                f"{gap:>+12.1%}  {'↑ better' if gap > 0 else '↓ worse'}"
            )
        else:
            print(f"{source:<18} {'N/A':>13} {mhj_acc:>13.1%} {'N/A':>13}")
else:
    print("(eval_df not available — run Phase 2 evaluation cells first)")
    for source in SOURCES:
        nb04_acc = NB04_REFERENCE.get(source)
        ref_str  = f"{nb04_acc:.1%}" if nb04_acc is not None else "N/A"
        print(f"{source:<18} {ref_str:>13} {'?':>13} {'?':>13}")

print()
print("Interpretation guide:")
print("  Positive transfer gap → centroid detects human jailbreaks better than")
print("  framework-generated ones (suggests human attacks are more stereotyped).")
print("  Negative gap → centroid is tuned to framework-specific patterns that")
print("  don't appear in human conversations (limited generalization).")
print("  Near-chance accuracy (≈50%) → centroid does not transfer to MHJ at all.")
print()
print("Reference accuracies (nb04, delta scorer, threshold=0, test split):")
for s, v in NB04_REFERENCE.items():
    print(f"  {s:<18} {str(v) if v else 'N/A'}")